## Usernames filter

This notebook aims to filter the raw usernames contained in `usernames.txt`, to keep only those created between March, 17th and September, 17th, and who let public their production.

### Chose subset of the total `usernames.txt`

In [30]:
X = 2 # Or 2 or 3

In [31]:
with open('usernames.txt', 'r', encoding='utf-8') as f:
    raw_names = f.readlines()

breakpoint = int(len(raw_names)/3)
batch = raw_names[breakpoint*(X-1):breakpoint*(X)]
if X == 3:
    batch += raw_names[breakpoint*(X)+1:]

with open(f'covid_users_{X}.txt', 'r', encoding='utf-8') as f:
    last_done = f.readlines()[-1]
    if "Last done:" in last_done:
        last_done = last_done.strip('Last done: ')
        batch = batch[batch.index(last_done)+1 : ]
    else : 
        print('No past attempt saved.')

print(f'Remaining length batch: {len(batch)}')

Remaining length batch: 30407


### Scrap Reddit to filter each username in the batch

In [ ]:
import requests
from datetime import datetime
import time

headers = {
    "User-Agent": "Mozilla/5.0 (compatible; scraper/1.0)"
}
session = requests.Session()
session.headers.update(headers)


start = datetime(2020, 3, 17)
end = datetime(2020, 9, 17)

to_save = []
missing = 0

for i, username in enumerate(batch):
    if i%20==0:
        print(f'Step {i}')
    time.sleep(0.5)
    try:
        r = session.get(
            f"https://www.reddit.com/user/{username.strip('\n')}/about.json", 
            timeout=3)
        r.raise_for_status()
        created_utc = r.json()["data"]["created_utc"]
    except Exception:
        missing+=1
        if missing % 5 == 0: 
            percent = (missing * 100) / (i + 1)
            print(f"Missing: {percent:.2f}%")
        continue
    date_regis = datetime.utcfromtimestamp(created_utc)
    if start <= date_regis <= end:
        r = session.get(
            f"https://www.reddit.com/user/{username.strip('\n')}/.json",
            timeout=3)
        r.raise_for_status()
        data = r.json()
        if data['data']['children']:
            to_save.append(username)
            print(f'Found:{len(to_save)}, Among:{i+1}')

print(f'Missing:{missing}')

with open(f'covid_users_{X}.txt', 'w', encoding='utf-8') as f:
    for username in to_save:
        f.write(username)

Step 0


C:\Users\cfrou\AppData\Local\Temp\ipykernel_5816\3510080833.py:34: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  date_regis = datetime.utcfromtimestamp(created_utc)


Missing: 41.67%


KeyboardInterrupt: 

In [ ]:
with open(f'covid_users_{X}.txt', 'a', encoding='utf-8') as f:
    for user in to_save:
        f.write(user)
    f.write(f'Last done: {username}')